# 01 — Exploration du corpus, des résumés et des faits

**Projet** : Résumé de comptes-rendus d'intervention — encodeur-décodeur appris, baseline extractive publiée

## Objectifs pédagogiques

1. Décrire un corpus de comptes-rendus **et** les résumés qu'on attend de lui : longueurs, compression, vocabulaire, période.
1. Lire la table des faits annotés : sept types, une part saillante, un décor qui n'est jamais résumé.
1. Mesurer ce que le résumé de référence couvre **de lui-même** : le plafond de la tâche, avant tout modèle.
1. Comprendre pourquoi un ROUGE ne se lit jamais seul : il ne dit rien de la vérité d'une phrase, la couverture des faits si.

## 0. Environnement

Toute la configuration vient de **Hydra** (`conf/`) : aucune valeur métier n'est codée en dur dans ce notebook. Si le corpus du notebook manque, le générateur synthétique le prend le relais (voir `make data` pour le corpus de référence).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (200 comptes-rendus), un budget
# d'époques réduit et une architecture réduite : l'exécution reste de quelques secondes, et les
# valeurs absolues d'un notebook ne sont **pas** celles de la référence publiée par le README.
NB_DOCUMENTS = 200
NB_EPOCHS = 8

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                f"train.epochs={NB_EPOCHS}",
                "model.params.layers=1",
                "model.params.units=96",
                "model.params.pretraining.epochs=1",
                "+model.params.vocab_size=800",
                "+model.params.min_frequency=2",
                "train.callbacks.progress_bar=false",
                "train.callbacks.logging_every=100",
                "train.early_stopping.enabled=false",
                "seed=42",
                "log_level=WARNING",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Bac à sable : les notebooks génèrent **leur** corpus (réduit) et écrivent **leurs** artefacts sous
# `outputs/notebooks/`. `data/raw`, `artifacts/` et donc les chiffres publiés par le README restent
# ceux de `make data` / `make train` : aucun notebook ne peut les modifier par accident.
NB_ROOT = PROJECT_ROOT / "outputs" / "notebooks"
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=NB_ROOT / "data",
    artifacts_dir=NB_ROOT / "artifacts",
).ensure()

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
ID_COLUMN = str(MODEL_NODE.get("id_column", "doc_id"))
COMPRESSION = float(MODEL_NODE.get("params", {}).get("compression", 0.45))
MAX_OUTPUT_TOKENS = int(MODEL_NODE.get("params", {}).get("max_output_tokens", 90))
STRATEGIES = list(MODEL_NODE.get("strategies_to_compare", []) or ["lead"])

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.45)")
print(f"Algorithme servi  : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Stratégies comparées : {', '.join(STRATEGIES)}")
print(f"Budget de longueur: compression {COMPRESSION:.2f}, plafond {MAX_OUTPUT_TOKENS} tokens")
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticSummaryCorpusGenerator
from src.data.loaders import SummaryCorpusLoader

LOADER = SummaryCorpusLoader(
    NB_PATHS,
    dataset_name=str(CONFIG.data.dataset_name),
    formats=CONFIG.data.formats,
    validation_enabled=CONFIG.data.validation.raw,
    lazy_validation=CONFIG.data.validation.lazy,
)
if LOADER.documents_path.exists():
    DOCUMENTS, REFERENCES, FACTS = LOADER.load_corpus()
    METADATA = LOADER.load_metadata()
    print(
        f"Corpus du notebook lu : {len(DOCUMENTS)} comptes-rendus, "
        f"{len(REFERENCES)} résumés de référence, {len(FACTS)} faits annotés"
    )
else:
    # Un clone frais n'a pas encore de corpus réduit : le notebook le génère, de façon déterministe,
    # dans son bac à sable. `make data` produit le corpus de référence (600 comptes-rendus) dans
    # `data/raw` — les deux ne se mélangent jamais.
    bundle = SyntheticSummaryCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, REFERENCES, FACTS = bundle.documents, bundle.queries, bundle.facts
    METADATA = bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_references(REFERENCES)
    LOADER.save_facts(FACTS)
    LOADER.save_metadata(METADATA)
    print(f"Corpus réduit généré dans outputs/notebooks/data : {len(DOCUMENTS)} comptes-rendus")

TRAIN_DOCUMENTS, TRAIN_REFERENCES, TRAIN_FACTS = LOADER.load_split("train")
VAL_DOCUMENTS, VAL_REFERENCES, VAL_FACTS = LOADER.load_split("val")
TEST_DOCUMENTS, TEST_REFERENCES, TEST_FACTS = LOADER.load_split("test")
SPLIT_SIZES = LOADER.split_sizes()
# Le test ne sert **jamais** à régler quoi que ce soit : les réglages se lisent sur la validation,
# le verdict sur le test, une seule fois.
GOLDEN = dict(zip(REFERENCES["doc_id"], REFERENCES["summary"], strict=False))
print("splits :", SPLIT_SIZES)
print("compression moyenne demandée :", round(float(REFERENCES["compression"].mean()), 4))

## 1. Les comptes-rendus

Un compte-rendu d'intervention raconte une visite : un équipement, un symptôme, une cause, des actions, parfois une pièce, une durée et un statut. Les quatre lectures ci-dessous suffisent à détecter les deux pathologies d'un corpus de résumé : une catégorie qui écrase toutes les autres, et des documents si courts que la compression n'a plus rien à couper.

In [ ]:
DOCUMENTS.head(3)[
    ["doc_id", "intervention_type", "urgency", "site", "n_sentences", "n_tokens", "split"]
]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
DOCUMENTS["intervention_type"].value_counts().plot.bar(
    ax=axes[0], title="Comptes-rendus par type d'intervention", rot=30
)
DOCUMENTS["urgency"].value_counts().plot.bar(ax=axes[1], title="Par urgence", rot=0)
DOCUMENTS["n_tokens"].plot.hist(ax=axes[2], bins=30, title="Longueur des documents (tokens)")
plt.tight_layout()
plt.show()

print(
    "période couverte        :",
    DOCUMENTS["published_at"].min().date(),
    "->",
    DOCUMENTS["published_at"].max().date(),
)
print("identifiants dupliqués  :", int(DOCUMENTS["doc_id"].duplicated().sum()))
print(
    "phrases par document    :",
    round(float(DOCUMENTS["n_sentences"].mean()), 1),
    f"(min {int(DOCUMENTS['n_sentences'].min())}, max {int(DOCUMENTS['n_sentences'].max())})",
)
print("tokens par document     :", round(float(DOCUMENTS["n_tokens"].mean()), 1))
print("sites distincts         :", DOCUMENTS["site"].nunique())

**Ce qu'il faut retenir**

- Un type d'intervention qui concentre la majorité des documents oriente tout ce que le modèle apprendra : la ventilation par type est un prérequis à la lecture des scores, et le notebook 06 ventile les erreurs par segment.
- La longueur des documents fixe la difficulté réelle : plus un compte-rendu est long, plus il contient de faits non saillants qu'un résumé doit **laisser tomber**.
- Des identifiants dupliqués casseraient toutes les jointures entre documents, résumés et faits : le contrat les refuse (notebook 02) et cette cellule les compte.

## 2. Les résumés de référence

Un résumé de référence n'est pas « le document en plus court » : il est **écrit** à partir des faits saillants, avec ses propres formulations. La compression qu'il atteint est la cible réelle de la tâche — celle que le budget de longueur du modèle doit respecter.

In [ ]:
from src.features.build_features import SentenceFeatureBuilder

profile = SentenceFeatureBuilder.profile(DOCUMENTS, REFERENCES)
display(pd.Series(profile, name="valeur").to_frame().round(4))

display(REFERENCES.head(3)[["doc_id", "n_sentences", "n_tokens", "n_salient_facts", "compression"]])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
REFERENCES["n_tokens"].plot.hist(ax=axes[0], bins=30, title="Longueur des résumés (tokens)")
REFERENCES["compression"].plot.hist(ax=axes[1], bins=30, title="Compression des résumés")
REFERENCES["n_salient_facts"].plot.hist(ax=axes[2], bins=12, title="Faits saillants par résumé")
plt.tight_layout()
plt.show()

print("compression moyenne     :", round(float(REFERENCES["compression"].mean()), 4))
print("compression maximale    :", round(float(REFERENCES["compression"].max()), 4))
print(
    "part au plafond du budget (>= 0,599) :",
    round(float((REFERENCES["compression"] >= 0.599).mean()), 3),
)
print(
    "phrases par résumé      :",
    round(float(REFERENCES["n_sentences"].mean()), 1),
    f"(min {int(REFERENCES['n_sentences'].min())}, max {int(REFERENCES['n_sentences'].max())})",
)

In [ ]:
# La question qui décide de tout : la référence **recopie-t-elle** le document, ou le réécrit-elle ?
# Si elle recopiait, un extracteur de phrases gagnerait ; elle réécrit, donc il faut générer.
example_id = str(DOCUMENTS.iloc[0]["doc_id"])
document = DOCUMENTS.loc[DOCUMENTS["doc_id"] == example_id, "text"].iloc[0]
reference = REFERENCES.loc[REFERENCES["doc_id"] == example_id, "summary"].iloc[0]
print(f"--- document {example_id} ({len(document.split())} mots) ---")
print(document)
print(f"\n--- résumé de référence ({len(reference.split())} mots) ---")
print(reference)
print(
    "\nphrases du document présentes telles quelles dans le résumé :",
    int(
        sum(
            1
            for sentence in reference.split(". ")
            if sentence.strip().rstrip(".") + "." in document
        )
    ),
)

**Ce qu'il faut retenir**

- La compression demandée est le premier chiffre à connaître : un budget de longueur fixé au-dessus d'elle produit des résumés plus longs que la référence, donc un rappel de ROUGE plafonné.
- Très peu de références touchent le plafond du budget : le corpus n'est pas un exercice de troncature, c'est un exercice de sélection **et** de reformulation.
- Les résumés de référence ne recopient pas les phrases du document : c'est la part de réécriture du corpus, et c'est elle qui empêche un extracteur d'atteindre 1,0.

## 3. Les faits annotés

Chaque phrase qui porte un fait rapportable a écrit une ligne dans la table des faits : son type, sa valeur, sa **surface** exacte dans le texte et l'index de sa phrase. Tous les faits ne sont pas saillants : le résumé de référence n'en rapporte qu'une partie, et c'est cette partie-là qui est mesurée.

In [ ]:
from src.data.schemas import fact_columns, salient_fact_counts

display(pd.Series(salient_fact_counts(FACTS), name="faits saillants").to_frame())
print("faits annotés          :", len(FACTS), f"({len(FACTS) / len(DOCUMENTS):.2f} par document)")
print(
    "faits saillants        :",
    int(FACTS["salient"].sum()),
    f"({FACTS['salient'].mean():.2f} par fait annoté)",
)
print("colonnes de fidélité publiées :", fact_columns())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
FACTS["fact_type"].value_counts().plot.bar(ax=axes[0], title="Faits annotés par type", rot=30)
(
    FACTS[FACTS["salient"] == 1]["fact_type"].value_counts() / FACTS["fact_type"].value_counts()
).plot.bar(ax=axes[1], title="Part saillante par type", rot=30)
plt.tight_layout()
plt.show()

example_facts = FACTS[FACTS["doc_id"] == example_id]
display(example_facts[["fact_type", "value", "surface", "salient", "sentence_index"]])

## 4. Le décor est un distracteur déclaré

Numéros de ticket, de téléphone ou de parking, horaires de présence, formules administratives : ces valeurs sont **dans** les documents et ne figurent **jamais** dans les résumés. Elles servent à piéger deux comportements : la recopie intégrale et l'invention. Le rapport publie les valeurs non supportées, donc un modèle qui en produit une est détecté (notebook 06).

In [ ]:
from src.data.vocabulary import DISTRACTORS, MAX_COMPRESSION, SUMMARY_SENTENCE_RANGE

corpus_text = " ".join(DOCUMENTS["text"])
reference_text = " ".join(REFERENCES["summary"])
rows = [
    {
        "distracteur": value,
        "dans les documents": int(corpus_text.count(value)),
        "dans les résumés": int(reference_text.count(value)),
    }
    for value in DISTRACTORS
]
display(pd.DataFrame(rows))
print("types de faits      :", sorted(FACTS["fact_type"].unique()))
print("phrases par résumé  :", SUMMARY_SENTENCE_RANGE, "(contrat)")
print("compression maximale:", MAX_COMPRESSION, "(plafond du générateur de corpus)")

## 5. Ce que la référence couvre d'elle-même

Avant de mesurer un modèle, on mesure la **vérité** : le résumé de référence contient-il les faits saillants qu'il est censé rapporter ? Si la réponse n'était pas « tous », aucune couverture de modèle ne serait interprétable — l'annotation et la référence se contrediraient. C'est le plafond de la tâche, et il se vérifie.

In [ ]:
from src.training.metrics import coverage_scores

per_document = []
for row in REFERENCES.itertuples(index=False):
    facts = FACTS[FACTS["doc_id"] == str(row.doc_id)]
    score = coverage_scores(str(row.summary), facts)
    per_document.append(
        {
            "doc_id": str(row.doc_id),
            "coverage": score["fact_coverage"],
            "unsupported": score["n_unsupported"],
        }
    )
CEILING = pd.DataFrame(per_document)
print(
    "couverture de la référence : min",
    round(float(CEILING["coverage"].min()), 4),
    "| moyenne",
    round(float(CEILING["coverage"].mean()), 4),
)
print("valeurs non supportées     :", int(CEILING["unsupported"].sum()))
print(
    "documents parfaitement couverts :", int((CEILING["coverage"] == 1.0).sum()), "/", len(CEILING)
)

In [ ]:
# Le plancher symétrique : un résumé vide a une couverture nulle et un ROUGE de 0,0 — la métrique
# est *définie* sur un système muet, ce qui est la condition pour publier un verdict.
from src.evaluation.rouge import rouge_scores
from src.training.metrics import trivial_floor

empty = rouge_scores(GOLDEN[str(REFERENCES.iloc[0]["doc_id"])], "")
copy_rouge = rouge_scores(
    GOLDEN[str(REFERENCES.iloc[0]["doc_id"])],
    DOCUMENTS.loc[DOCUMENTS["doc_id"] == REFERENCES.iloc[0]["doc_id"], "text"].iloc[0],
)
display(
    pd.DataFrame(
        [
            trivial_floor(),
            {"rouge1_f": empty.rouge1.f1, "rouge_l_f": empty.rouge_l.f1},
            {"rouge1_f": copy_rouge.rouge1.f1, "rouge_l_f": copy_rouge.rouge_l.f1},
        ],
        index=["résumé vide (plancher)", "résumé vide (ROUGE)", "recopie intégrale (exemple)"],
    ).round(4)
)

**Ce qu'il faut retenir**

- Un plafond mesuré avant tout modèle est ce qui rend un score interprétable : ici la référence couvre exactement les faits saillants qu'elle annonce.
- Un ROUGE élevé sur une recopie intégrale serait un mauvais signe : il signifierait que le corpus ne demande pas de réécriture — ce n'est pas ce corpus.
- La couverture et le ROUGE mesurent deux choses différentes, et le projet publie les deux : le premier dit si le résumé est **vrai**, le second s'il ressemble à la référence.